# Intro. ao Aprendizado de Máquina (ENGG67) - Avaliação 1

Professor: ANTONIO CARLOS LOPES FERNANDES JUNIOR<br>
Alunos: ANDRE NEVES DA ROCHA CASTRO e HUGO SOUZA ROCHA

##  Caracterização e Exploração Inicial

O [Heart Disease Dataset](https://archive.ics.uci.edu/ml/datasets/Heart+Disease) é um conjunto de dados que reúne informações clínicas e resultados de exames de pacientes. Os dados incluem variáveis como idade, sexo, tipo de dor no peito, pressão arterial, colesterol e resultados de testes cardíacos, permitindo análises preditivas relevantes na área da saúde.

In [10]:
from ucimlrepo import fetch_ucirepo 
  
heart_disease = fetch_ucirepo(id=45) 
  
X = heart_disease.data.features 
y = heart_disease.data.targets 

# Verificando as dimensões de X (preditores) e y (alvo)
n_instancias, n_atributos = X.shape
n_alvos = y.shape[1] if len(y.shape) > 1 else 1

print(f"Número de instâncias (pacientes): {n_instancias}")
print(f"Número de atributos preditivos: {n_atributos}")
print(f"Número de atributos alvo: {n_alvos}")

Número de instâncias (pacientes): 303
Número de atributos preditivos: 13
Número de atributos alvo: 1


A classificação do dataset como **multivariado** justifica-se pelo fato de que cada observação (paciente) é composta por múltiplas variáveis medidas simultaneamente (como idade, colesterol, pressão arterial, etc.), as quais, em conjunto, fornecem o contexto necessário para a predição. No contexto de Aprendizado de Máquina, identificar corretamente os **13 atributos preditivos** preditivos é essencial, pois eles representam o espaço de características que o modelo explorará para encontrar padrões. O atributo alvo (num), que indica a presença ou ausência de doença cardíaca.

### Caracterização dos Dados: Tipo e Escala

Antes de selecionar algoritmos de aprendizado de máquina ou técnicas de pré-processamento, é essencial realizar a **categorização dos dados**. Esta etapa distingue a forma como o computador armazena a informação (tipo técnico) de como o dado se comporta matematicamente (natureza estatística).

Os atributos do *dataset* são divididos em duas grandes classes:

#### 1. Atributos Qualitativos (Categóricos)
Representam qualidades ou categorias. Não possuem valor numérico intrínseco, mesmo quando codificados com números.

* **Escala Nominal** (Sem ordem intrínseca):
    * `sex`: Sexo do paciente (1 = masculino; 0 = feminino).
    * `cp`: Tipo de dor no peito (4 valores distintos).
    * `fbs`: Açúcar no sangue em jejum > 120 mg/dl (Binário: 1 ou 0).
    * `restecg`: Resultados eletrocardiográficos em repouso.
    * `exang`: Angina induzida por exercício (1 = sim; 0 = não).
    * `thal`: Diagnóstico de cintilografia (normal, defeito fixo, defeito reversível).
* **Escala Ordinal** (Possuem uma ordem ou hierarquia lógica):
    * `slope`: A inclinação do segmento ST no pico do exercício (pode ser interpretada como uma progressão de intensidade).
    * `num`: O diagnóstico de doença cardíaca (0 para ausência, 1 a 4 para níveis de severidade).

---

#### 2. Atributos Quantitativos (Numéricos)
Representam medidas reais onde as operações aritméticas (soma, média) fazem sentido.

* **Escala de Razão** (O zero é absoluto e indica ausência da medida):
    * `age`: Idade (em anos).
    * `trestbps`: Pressão arterial em repouso (mmHg).
    * `chol`: Colesterol sérico (mg/dl).
    * `thalach`: Frequência cardíaca máxima atingida.
    * `oldpeak`: Depressão de ST induzida pelo exercício (valor contínuo).
    * `ca`: Número de vasos principais coloridos por fluoroscopia (embora seja discreto/inteiro, representa uma contagem física).

In [14]:
print("Informações detalhadas das variáveis (Metadados):")
display(heart_disease.variables[['name', 'type', 'description']])

Informações detalhadas das variáveis (Metadados):


,name,type,description
0,age,Integer,None
1,sex,Categorical,None
2,cp,Categorical,None
3,trestbps,Integer,resting blood pressure (on admission to the ho...
4,chol,Integer,serum cholestoral
5,fbs,Categorical,fasting blood sugar > 120 mg/dl
6,restecg,Categorical,None
7,thalach,Integer,maximum heart rate achieved
8,exang,Categorical,exercise induced angina
9,oldpeak,Integer,ST depression induced by exercise relative to ...


### Balanceamento e Estatística Univariada

#### Analise de Balanceamento:

In [18]:
print("Distribuição das classes no atributo alvo:")
print(y.value_counts())

Distribuição das classes no atributo alvo:
num
0      164
1       55
2       36
3       35
4       13
Name: count, dtype: int64


#### Caracterização Estatística:

In [19]:
estatisticas = X.describe().T  # Gera média, desvio-padrão, min, quartis e max
estatisticas['moda'] = X.mode().iloc[0]
estatisticas['obliquidade'] = X.skew()
estatisticas['curtose'] = X.kurtosis()

estatisticas = estatisticas.rename(columns={
    'mean': 'média', 'std': 'desvio_padrão', '50%': 'mediana', 
    '25%': 'Q1', '75%': 'Q3'
})

display(estatisticas[['média', 'mediana', 'moda', 'desvio_padrão', 'Q1', 'Q3', 'obliquidade', 'curtose']])

,média,mediana,moda,desvio_padrão,Q1,Q3,obliquidade,curtose
age,54.438944,56.0,58.0,9.038662,48.0,61.0,-0.209060,-0.523383
sex,0.679868,1.0,1.0,0.467299,0.0,1.0,-0.774935,-1.408819
cp,3.158416,3.0,4.0,0.960126,3.0,4.0,-0.841754,-0.400655
trestbps,131.689769,130.0,120.0,17.599748,120.0,140.0,0.706035,0.880074
chol,246.693069,241.0,197.0,51.776918,211.0,275.0,1.135503,4.491724
fbs,0.148515,0.0,0.0,0.356198,0.0,0.0,1.986652,1.959678
restecg,0.990099,1.0,0.0,0.994971,0.0,2.0,0.019900,-1.999331
thalach,149.607261,153.0,162.0,22.875003,133.5,166.0,-0.537449,-0.053541
exang,0.326733,0.0,0.0,0.469794,0.0,1.0,0.742532,-1.458317
oldpeak,1.039604,0.8,0.0,1.161075,0.0,1.6,1.269720,1.575813
